# 07 Gaussian Naive Bayes With Basic Image Features

This notebook trains only `GaussianNB` using the existing basic image-derived features. Gaussian Naive Bayes is a probability-based supervised classifier. It assumes features are conditionally independent and approximately normally distributed within each class, which is probably weak for image features, so this is mainly a simple baseline.

## 1. Project setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

## 2. Imports

In [ ]:
from src.extract_basic_features import BASIC_FEATURE_COLUMNS
from src.train_logistic_regression import FEATURES_CSV, load_features
from src.train_gaussian_nb import (
    SCORES_CSV,
    CONFUSION_MATRIX_FIGURE,
    build_comparison_table,
    train_gaussian_nb,
)

## 3. Paths and configuration

In [ ]:
print(f"Basic features CSV: {FEATURES_CSV}")
print(f"Scores CSV: {SCORES_CSV}")
print(f"Confusion matrix figure: {CONFUSION_MATRIX_FIGURE}")

## 4. Load metadata

In [ ]:
features = load_features(FEATURES_CSV)
features.head()

## 5. Sanity checks

In [ ]:
print(f"Rows: {len(features)}")
print("Label counts:")
print(features["label"].value_counts().sort_index())
print(f"Image feature columns: {len(BASIC_FEATURE_COLUMNS)}")
print(BASIC_FEATURE_COLUMNS)
assert set(BASIC_FEATURE_COLUMNS).issubset(features.columns)
assert features["area_group"].notna().all()

## 6. Main analysis

Only image-derived feature columns are used as predictors. Metadata columns are used for labels, auditing, and grouped splitting, not as model inputs.

In [ ]:
scores, report, matrix, figure_path, y_train, y_test = train_gaussian_nb(features)
scores

## 7. Results/figures

Balanced accuracy and macro F1 matter more than raw accuracy because the dataset has many more ordered images than disordered images.

In [ ]:
scores.to_csv(SCORES_CSV, index=False)

print("Selected split label counts:")
print("Train:")
print(y_train.value_counts().sort_index())
print("Test:")
print(y_test.value_counts().sort_index())
print("\nPer-class precision/recall/F1:")
print(report)
print("Confusion matrix rows=true, columns=predicted:")
print(matrix)
print("\nModel comparison:")
print(build_comparison_table(scores))
print(f"\nSaved scores to {SCORES_CSV}")
print(f"Saved confusion matrix figure to {figure_path}")

## 8. Notes for report

- GaussianNB is a probability-based classifier.
- It assumes feature independence and normal numeric feature distributions within each class.
- This assumption is probably weak for image features, so GaussianNB is mainly a simple baseline.
- Compare it using macro F1 and balanced accuracy, not raw accuracy, because the dataset is imbalanced.
- This step does not add SVC, KNN, HOG, graph features, augmentation, or CNNs.